In [ ]:
from transformers import pipeline

ner_pipeline = pipeline("ner", model="ab-ai/pii_model_based_on_distilbert")

def redact_pii(text):
    entities = predict_entities_chunked(text)
    for entity in sorted(entities, key=lambda x: x['start'], reverse=True):
        text = text[:entity['start']] + "[REDACTED]" + text[entity['end']:]
    return text

data['Resume_anonymized'] = data['Resume'].apply(redact_pii)
data[["Resume_anonymized"]].to_csv("raw_data/resumes_anonymized.csv", index=False)

def run_bow(text_series, label):
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
    X = vectorizer.fit_transform(text_series + " " + data['Job Description'])
    y = data['Hired']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    feature_names = vectorizer.get_feature_names_out()
    coefficients = model.coef_[0]

    print(f"\n=== BoW results: {label} ===")
    top_positive = np.argsort(coefficients)[-20:]
    print("Words that increase hiring chances:")
    for i in top_positive:
        print(f"  {feature_names[i]:30s} {coefficients[i]:.3f}")

    top_negative = np.argsort(coefficients)[:20]
    print("\nWords that decrease hiring chances:")
    for i in top_negative:
        print(f"  {feature_names[i]:30s} {coefficients[i]:.3f}")

run_bow(data['Resume'], "BEFORE anonymization")
run_bow(data['Resume_anonymized'], "AFTER anonymization")